# Module 1 : Ma première classification

**Durée** : environ 2 h 30, en deux notebooks. **Algorithme** : la régression logistique.

À la fin de ce module, tu sauras :

1. expliquer ce qu'est un problème de classification (variables explicatives, cible) ;
2. expliquer pourquoi on sépare les données en jeu d'entraînement et jeu de test ;
3. entraîner une régression logistique et obtenir une prédiction ;
4. lire une matrice de confusion et calculer l'accuracy et le rappel ;
5. reproduire en code un résultat obtenu dans DATAPULSE.

| Étape | Contenu | Durée |
|---|---|---|
| A | Leçon et mini-quiz | 15 min |
| B | Manipulation dans DATAPULSE | 30 min |
| C | Lire le code | 15 min |
| D | Code à trous (3 exercices) | 30 min |
| E à G | Défi, comparaison et bilan : notebook `module1_defi.ipynb` | 70 min |

**Comment utiliser ce notebook** : clique dans une cellule grise, puis appuie sur **Maj + Entrée** pour l'exécuter. Exécute les cellules **dans l'ordre**, de haut en bas.

Commence par la cellule ci-dessous : elle charge les outils du module (quiz, vérifications, indices).

In [ ]:
import numpy, pandas, sklearn   # dans le navigateur, le premier chargement prend un moment
from module1_outils import *
demarrer()

---
## Étape A : Leçon (15 min)

### Qu'est-ce qu'un modèle ?

Un **modèle** est une fonction apprise à partir d'exemples. On lui montre des centaines de cas dont on connaît la réponse ; il en tire une règle qu'il applique ensuite à des cas nouveaux.

### Variables explicatives et variable cible

Notre jeu de données décrit **569 tumeurs du sein**. Pour chacune :

- **30 mesures** prises sur une image au microscope (rayon, texture, périmètre…) : ce sont les **variables explicatives**, notées `X` ;
- le **diagnostic** : maligne (`0`) ou bénigne (`1`). C'est la **cible**, notée `y` : ce que le modèle doit prédire.

```
        X (30 mesures)                         y (cible)
 rayon  texture  périmètre  …          →      diagnostic
 17,99   10,38    122,8     …          →      0 (maligne)
 13,54   14,36     87,46    …          →      1 (bénigne)
```

Prédire une **catégorie** (maligne ou bénigne) s'appelle une **classification**.

### Pourquoi un jeu de test ?

Un modèle qui récite ses exemples ne prouve rien, comme un élève qui apprend par cœur les corrigés. On met donc de côté une partie des données, **le jeu de test**, que le modèle ne voit jamais pendant l'apprentissage. Son score sur ce jeu dit s'il sait **généraliser**.

```
 569 tumeurs ──┬── 80 % → jeu d'entraînement (455) : le modèle apprend
               └── 20 % → jeu de test       (114) : on évalue
```

### La régression logistique en une idée

Elle estime, pour chaque tumeur, une **probabilité** d'appartenir à une classe (par exemple 0,93 pour « bénigne »). Puis elle applique un **seuil** : au-dessus de 0,5, elle prédit « bénigne » ; en dessous, « maligne ».

### Mini-quiz

Exécute la cellule suivante pour voir les trois questions.

In [ ]:
quiz()

Remplace `"?"` par ta réponse (`"a"`, `"b"` ou `"c"`) et exécute. Tu peux réessayer, mais seule ta première réponse compte pour le score.

In [ ]:
repondre(1, "?")

In [ ]:
repondre(2, "?")

In [ ]:
repondre(3, "?")

---
## Étape B : Manipulation dans DATAPULSE (30 min)

Ouvre **[DATAPULSE](https://datapulse-t69vfezujmrcobcdsgrpt3.streamlit.app/)** dans un nouvel onglet. Les données du module sont dans le fichier `cancer_du_sein.csv`, à côté de ce notebook (clic droit sur le fichier, puis *Download*).

1. **📂 Chargement & EDA** : charge `cancer_du_sein.csv`.
2. Observe les statistiques descriptives et la distribution de la colonne `target`. Y a-t-il autant de tumeurs malignes que bénignes ?
3. **🎯 Modélisation Supervisée** : choisis *Classification*, la cible `target`, puis *Régression Logistique*.
4. Garde les réglages par défaut (test 20 %, graine 42), lance l'entraînement, et note l'accuracy et la matrice de confusion.
5. **Expériences guidées** : refais l'entraînement en changeant un seul réglage à la fois, et complète le tableau (double-clique dessus pour le modifier).
6. **🧪 Test du Modèle** : saisis des valeurs et lis la probabilité prédite.

| Test (%) | Graine | Accuracy |
|---|---|---|
| 20 | 42 | |
| 10 | 42 | |
| 40 | 42 | |
| 20 | 7 | |
| 20 | 123 | |

**Questions de réflexion** (sans note). Écris ta réponse entre les triples guillemets, puis exécute.

In [ ]:
reflexion("graine", """
Les résultats changent-ils avec la graine ? Qu'est-ce que cela dit de la fiabilité d'un seul découpage ?
Ta réponse :
""")

In [ ]:
reflexion("erreur_grave", """
Pour le cancer du sein, quelle erreur est la plus grave : un faux positif ou un faux négatif ? Pourquoi ?
Ta réponse :
""")

---
## Étape C : Lire le code (15 min)

Voici le code scikit-learn qui reproduit **exactement** l'analyse que tu viens de faire dans DATAPULSE. Lis-le ligne par ligne. Dans DATAPULSE, le bouton **💻 Montre-moi le code** affiche un code équivalent pour n'importe quelle analyse.

```python
# 1. Charger les données
from sklearn.datasets import load_breast_cancer
X, y = load_breast_cancer(return_X_y=True, as_frame=True)

# 2. Séparer entraînement / test (même réglage que DATAPULSE)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3. Pipeline : normalisation + modèle (la normalisation est
#    ajustée sur le train uniquement, pour éviter toute fuite)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

modele = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

# 4. Entraîner
modele.fit(X_train, y_train)

# 5. Évaluer sur le jeu de test
from sklearn.metrics import accuracy_score, confusion_matrix
y_pred = modele.predict(X_test)
print("Accuracy :", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
```

Réfléchis à chaque question avant d'ouvrir la réponse.

<details><summary><b>Pourquoi <code>stratify=y</code> ?</b></summary>

Il y a 212 tumeurs malignes pour 357 bénignes (37 % / 63 %). Sans stratification, le hasard peut mettre trop de tumeurs malignes d'un côté. `stratify=y` garde les mêmes proportions dans le jeu d'entraînement et dans le jeu de test.
</details>

<details><summary><b>Pourquoi <code>random_state=42</code> ?</b></summary>

Le découpage est tiré au hasard. Fixer la graine rend ce hasard **reproductible** : même graine, mêmes lignes de test, mêmes résultats. C'est ce qui permet de comparer ton code avec DATAPULSE. Le nombre 42 n'a rien de spécial.
</details>

<details><summary><b>Pourquoi la normalisation est-elle dans le <code>Pipeline</code> ?</b></summary>

Les mesures n'ont pas la même échelle (une aire vaut des centaines, une texture quelques unités). `StandardScaler` les ramène à une échelle comparable, ce qui aide la régression logistique. Placé dans le `Pipeline`, il calcule ses moyennes **sur le jeu d'entraînement seulement**. S'il les calculait sur toutes les données, des informations du jeu de test « fuiraient » dans l'apprentissage : c'est une **fuite de données**, et le score obtenu serait trop optimiste.
</details>

---
## Étape D : Code à trous (30 min)

Trois exercices. Dans chacun, remplace les `...` par ton code, exécute la cellule, puis la cellule `verifier(...)`.

- Bloqué ? `indice(...)` donne un indice, du plus vague au plus précis (3 niveaux).
- La solution se débloque après 3 tentatives.

La cellule suivante charge les données : exécute-la.

In [ ]:
from sklearn.datasets import load_breast_cancer
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
print(X.shape, "→", X.shape[0], "tumeurs et", X.shape[1], "variables explicatives")

### D1 : Séparer entraînement et test

Consigne : **20 %** de données de test, découpage **stratifié** sur la cible, graine **42**.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=..., stratify=..., random_state=...
)

In [ ]:
verifier("D1")

In [ ]:
indice("D1")    # seulement si tu es bloqué

In [ ]:
solution("D1")  # disponible après 3 tentatives

### D2 : Créer et entraîner le modèle

Consigne : un `Pipeline` avec la normalisation (`StandardScaler`), puis une régression logistique (`max_iter=1000`, `random_state=42`), entraîné sur le jeu d'entraînement.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

modele = Pipeline([
    ("scaler", ...),   # la normalisation
    ("clf", ...),      # la régression logistique
])
modele.fit(..., ...)

In [ ]:
verifier("D2")

In [ ]:
indice("D2")    # seulement si tu es bloqué

In [ ]:
solution("D2")  # disponible après 3 tentatives

### D3 : Évaluer le modèle

Consigne : prédire sur le jeu de test, puis calculer :

- `accuracy` : la part de prédictions correctes ;
- `matrice` : la matrice de confusion ;
- `rappel` : la part des tumeurs **malignes** (classe `0`) que le modèle a bien détectées.

**Lire une matrice de confusion** : les lignes sont les vraies classes, les colonnes les prédictions.

```
                    prédit maligne   prédit bénigne
 vraie maligne            VP               FN        ← FN : cancer manqué !
 vraie bénigne            FP               VN
```

- **accuracy** = (VP + VN) / total
- **rappel** (des malignes) = VP / (VP + FN) : sur toutes les tumeurs malignes, combien le modèle en a trouvé
- **précision** (des malignes) = VP / (VP + FP) : quand le modèle dit « maligne », à quel point il a raison

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score

y_pred = modele.predict(...)
accuracy = accuracy_score(..., ...)
matrice = confusion_matrix(..., ...)
rappel = recall_score(..., ..., pos_label=0)

print("Accuracy :", accuracy)
print(matrice)
print("Rappel (tumeurs malignes) :", rappel)

In [ ]:
verifier("D3")

In [ ]:
indice("D3")    # seulement si tu es bloqué

In [ ]:
solution("D3")  # disponible après 3 tentatives

**Compare avec DATAPULSE** : ton accuracy et ta matrice sont-elles les mêmes que celles notées à l'étape B (test 20 %, graine 42) ? Elles devraient l'être exactement.

---
### Suite : le défi

Ouvre le notebook **`module1_defi.ipynb`**. Tu y écriras toute l'analyse seul, depuis une cellule vide, puis la plateforme comparera ton résultat à celui de DATAPULSE.